In [1]:
%%writefile governance/model_cards/generator.py
"""
Automated Model Card Generator
Enterprise MLOps Platform

Generates standardized documentation for every trained model.
No model enters production without a model card.
"""

import json
import mlflow
from mlflow.tracking import MlflowClient
from datetime import datetime


client = MlflowClient()


def generate_model_card(run_id: str, model_name: str = "fraud-detection-model") -> dict:
    """Generate a model card from an MLflow run."""
    
    run = client.get_run(run_id)
    params = run.data.params
    metrics = run.data.metrics
    
    card = {
        "model_card_version": "1.0",
        "generated_at": datetime.now().isoformat(),
        
        "model_details": {
            "name": model_name,
            "run_id": run_id,
            "framework": "XGBoost",
            "type": "Binary Classification — Fraud Detection",
            "parameters": params,
        },
        
        "intended_use": {
            "primary": "Real-time transaction fraud detection",
            "users": "Fraud detection platform, automated scoring pipeline",
            "out_of_scope": "Credit decisioning, customer segmentation, marketing",
        },
        
        "training_data": {
            "dataset": params.get("dataset", "unknown"),
            "features": int(params.get("num_features", 0)),
            "class_balance": f"Fraud rate ~0.4% (imbalanced)",
            "split_method": "Time-based (80/20) — no future data leakage",
        },
        
        "performance": {
            "auc_pr": metrics.get("auc_pr"),
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "f1_score": metrics.get("f1_score"),
            "true_positives": int(metrics.get("true_positives", 0)),
            "false_positives": int(metrics.get("false_positives", 0)),
            "false_negatives": int(metrics.get("false_negatives", 0)),
            "true_negatives": int(metrics.get("true_negatives", 0)),
        },
        
        "ethical_considerations": {
            "bias_testing": "pending",
            "fairness_metrics": "pending — required before production",
            "demographic_analysis": "pending",
        },
        
        "limitations": [
            "Trained on synthetic data — production deployment requires validation on real transactions",
            "Feature store latency assumptions based on local testing, not production infrastructure",
            "Fraud patterns limited to 5 typologies — real-world fraud is more diverse",
            "Geographic coverage limited to UK, US, India — model may underperform in other regions",
        ],
        
        "governance": {
            "regulatory_review": "pending",
            "bias_check_passed": False,
            "explainability_check_passed": False,
            "approved_for_production": False,
            "approved_by": None,
            "approval_date": None,
        },
    }
    
    return card


def save_model_card(card: dict, output_path: str = None):
    """Save model card as JSON."""
    if output_path is None:
        output_path = f"governance/model_cards/card_{card['model_details']['run_id'][:8]}.json"
    
    with open(output_path, 'w') as f:
        json.dump(card, f, indent=2)
    
    print(f"Model card saved: {output_path}")
    return output_path


def print_model_card(card: dict):
    """Print model card in readable format."""
    print(f"\n{'='*60}")
    print(f"MODEL CARD — {card['model_details']['name']}")
    print(f"{'='*60}")
    print(f"Generated: {card['generated_at']}")
    print(f"Run ID: {card['model_details']['run_id'][:8]}...")
    print(f"Framework: {card['model_details']['framework']}")
    
    print(f"\nINTENDED USE")
    print(f"  Primary: {card['intended_use']['primary']}")
    print(f"  Out of scope: {card['intended_use']['out_of_scope']}")
    
    print(f"\nTRAINING DATA")
    print(f"  Dataset: {card['training_data']['dataset']}")
    print(f"  Features: {card['training_data']['features']}")
    print(f"  Split: {card['training_data']['split_method']}")
    
    perf = card['performance']
    print(f"\nPERFORMANCE")
    print(f"  AUC-PR:    {perf['auc_pr']:.4f}")
    print(f"  Precision: {perf['precision']:.4f}")
    print(f"  Recall:    {perf['recall']:.4f}")
    print(f"  F1:        {perf['f1_score']:.4f}")
    print(f"  Caught: {perf['true_positives']} | Missed: {perf['false_negatives']} | False alarms: {perf['false_positives']}")
    
    print(f"\nETHICAL CONSIDERATIONS")
    for k, v in card['ethical_considerations'].items():
        print(f"  {k}: {v}")
    
    print(f"\nLIMITATIONS")
    for lim in card['limitations']:
        print(f"  - {lim}")
    
    gov = card['governance']
    print(f"\nGOVERNANCE STATUS")
    print(f"  Bias check:          {'PASSED' if gov['bias_check_passed'] else 'PENDING'}")
    print(f"  Explainability:      {'PASSED' if gov['explainability_check_passed'] else 'PENDING'}")
    print(f"  Production approved: {'YES' if gov['approved_for_production'] else 'NO'}")


if __name__ == "__main__":
    experiment = mlflow.get_experiment_by_name("fraud-detection-synthetic")
    runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.auc_pr DESC"])
    best_run_id = runs.iloc[0]['run_id']
    
    card = generate_model_card(best_run_id)
    print_model_card(card)
    save_model_card(card)

Writing governance/model_cards/generator.py


In [2]:
import mlflow
from governance.model_cards.generator import generate_model_card, print_model_card, save_model_card

# Generate card for our best synthetic model
experiment = mlflow.get_experiment_by_name("fraud-detection-synthetic")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.auc_pr DESC"])
best_run_id = runs.iloc[0]['run_id']

card = generate_model_card(best_run_id)
print_model_card(card)
save_model_card(card)


MODEL CARD — fraud-detection-model
Generated: 2026-03-07T15:22:45.434749
Run ID: 36af9fec...
Framework: XGBoost

INTENDED USE
  Primary: Real-time transaction fraud detection
  Out of scope: Credit decisioning, customer segmentation, marketing

TRAINING DATA
  Dataset: synthetic_500k
  Features: 33
  Split: Time-based (80/20) — no future data leakage

PERFORMANCE
  AUC-PR:    0.9359
  Precision: 0.8866
  Recall:    0.8866
  F1:        0.8866
  Caught: 0 | Missed: 0 | False alarms: 0

ETHICAL CONSIDERATIONS
  bias_testing: pending
  fairness_metrics: pending — required before production
  demographic_analysis: pending

LIMITATIONS
  - Trained on synthetic data — production deployment requires validation on real transactions
  - Feature store latency assumptions based on local testing, not production infrastructure
  - Fraud patterns limited to 5 typologies — real-world fraud is more diverse
  - Geographic coverage limited to UK, US, India — model may underperform in other regions

GOVE

'governance/model_cards/card_36af9fec.json'

In [3]:
%%writefile governance/bias_detection/bias_checker.py
"""
Bias Detection and Fairness Analysis
Enterprise MLOps Platform

Checks if the model's predictions are fair across demographic groups.
A model CANNOT reach production if bias thresholds are exceeded.
"""

import numpy as np
import pandas as pd


class BiasChecker:
    """
    Measures model fairness across protected groups.
    Uses disparate impact ratio and equalized odds.
    """
    
    def __init__(self, disparate_impact_threshold: float = 0.8,
                 equalized_odds_threshold: float = 0.1):
        self.di_threshold = disparate_impact_threshold
        self.eo_threshold = equalized_odds_threshold
    
    def check_bias(self, y_true: np.ndarray, y_pred: np.ndarray,
                   y_prob: np.ndarray, group_labels: np.ndarray,
                   group_name: str = "group") -> dict:
        """
        Run bias analysis across groups.
        
        Args:
            y_true: Actual labels (0/1)
            y_pred: Predicted labels (0/1)
            y_prob: Predicted probabilities
            group_labels: Group membership for each sample
            group_name: Name of the grouping variable
        
        Returns:
            dict with bias metrics and pass/fail status
        """
        groups = np.unique(group_labels)
        
        group_metrics = {}
        for g in groups:
            mask = group_labels == g
            n = mask.sum()
            if n < 10:
                continue
            
            g_true = y_true[mask]
            g_pred = y_pred[mask]
            g_prob = y_prob[mask]
            
            tp = ((g_pred == 1) & (g_true == 1)).sum()
            fp = ((g_pred == 1) & (g_true == 0)).sum()
            fn = ((g_pred == 0) & (g_true == 1)).sum()
            tn = ((g_pred == 0) & (g_true == 0)).sum()
            
            flag_rate = g_pred.mean()
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
            fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            avg_score = g_prob.mean()
            
            group_metrics[str(g)] = {
                "count": int(n),
                "fraud_count": int(g_true.sum()),
                "flag_rate": round(flag_rate, 6),
                "false_positive_rate": round(fpr, 6),
                "false_negative_rate": round(fnr, 6),
                "precision": round(precision, 4),
                "recall": round(recall, 4),
                "avg_fraud_score": round(avg_score, 6),
            }
        
        # Disparate Impact Ratio
        flag_rates = {g: m["flag_rate"] for g, m in group_metrics.items() if m["flag_rate"] > 0}
        if flag_rates:
            max_rate = max(flag_rates.values())
            min_rate = min(flag_rates.values())
            di_ratio = min_rate / max_rate if max_rate > 0 else 1.0
        else:
            di_ratio = 1.0
        
        # Equalized Odds — max difference in FPR and FNR across groups
        fprs = [m["false_positive_rate"] for m in group_metrics.values()]
        fnrs = [m["false_negative_rate"] for m in group_metrics.values()]
        eo_fpr_diff = max(fprs) - min(fprs) if fprs else 0
        eo_fnr_diff = max(fnrs) - min(fnrs) if fnrs else 0
        eo_max_diff = max(eo_fpr_diff, eo_fnr_diff)
        
        # Pass/fail
        di_passed = di_ratio >= self.di_threshold
        eo_passed = eo_max_diff <= self.eo_threshold
        overall_passed = di_passed and eo_passed
        
        result = {
            "group_name": group_name,
            "num_groups": len(group_metrics),
            "group_metrics": group_metrics,
            "disparate_impact_ratio": round(di_ratio, 4),
            "disparate_impact_threshold": self.di_threshold,
            "disparate_impact_passed": di_passed,
            "equalized_odds_fpr_diff": round(eo_fpr_diff, 4),
            "equalized_odds_fnr_diff": round(eo_fnr_diff, 4),
            "equalized_odds_max_diff": round(eo_max_diff, 4),
            "equalized_odds_threshold": self.eo_threshold,
            "equalized_odds_passed": eo_passed,
            "overall_passed": overall_passed,
        }
        
        return result
    
    def print_report(self, result: dict):
        """Print bias analysis report."""
        print(f"\nBIAS ANALYSIS — {result['group_name']}")
        print(f"{'='*60}")
        
        print(f"\n{'Group':<15} {'Count':>8} {'Fraud':>6} {'Flag%':>8} {'FPR':>8} {'FNR':>8} {'Recall':>8}")
        print(f"{'-'*63}")
        
        for g, m in result['group_metrics'].items():
            print(f"{g:<15} {m['count']:>8,} {m['fraud_count']:>6} {m['flag_rate']*100:>7.3f}% {m['false_positive_rate']*100:>7.3f}% {m['false_negative_rate']*100:>7.3f}% {m['recall']*100:>7.1f}%")
        
        print(f"\nDisparate Impact Ratio: {result['disparate_impact_ratio']:.4f} (threshold: >= {result['disparate_impact_threshold']})")
        print(f"  {'PASSED' if result['disparate_impact_passed'] else 'FAILED'}")
        
        print(f"\nEqualized Odds Max Diff: {result['equalized_odds_max_diff']:.4f} (threshold: <= {result['equalized_odds_threshold']})")
        print(f"  FPR diff: {result['equalized_odds_fpr_diff']:.4f}")
        print(f"  FNR diff: {result['equalized_odds_fnr_diff']:.4f}")
        print(f"  {'PASSED' if result['equalized_odds_passed'] else 'FAILED'}")
        
        print(f"\nOVERALL: {'*** PASSED ***' if result['overall_passed'] else '*** FAILED — BLOCKED FROM PRODUCTION ***'}")

Writing governance/bias_detection/bias_checker.py


In [5]:
from governance.bias_detection.bias_checker import BiasChecker
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import average_precision_score
import pandas as pd
import numpy as np

# Load and prepare data
df = pd.read_csv('data/raw/synthetic_transactions.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['cardholder_id', 'timestamp']).reset_index(drop=True)

# Quick feature engineering (same as before)
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['is_night'] = ((df['hour'] >= 0) & (df['hour'] < 6)).astype(int)
df['prev_timestamp'] = df.groupby('cardholder_id')['timestamp'].shift(1)
df['time_since_last_txn'] = (df['timestamp'] - df['prev_timestamp']).dt.total_seconds().fillna(-1)
df['is_rapid'] = (df['time_since_last_txn'].between(0, 300)).astype(int)
df['ch_avg_amount'] = df.groupby('cardholder_id')['amount'].transform(lambda x: x.expanding().mean().shift(1))
df['ch_std_amount'] = df.groupby('cardholder_id')['amount'].transform(lambda x: x.expanding().std().shift(1))
df['amount_vs_personal_avg'] = ((df['amount'] - df['ch_avg_amount']) / (df['ch_std_amount'] + 1e-6)).fillna(0)
df['amount_log'] = np.log1p(df['amount'])
df['is_round_amount'] = (df['amount'] % 10 == 0).astype(int)
df['date'] = df['timestamp'].dt.date
df['daily_txn_count'] = df.groupby(['cardholder_id', 'date'])['amount'].transform('count')
df['daily_spend'] = df.groupby(['cardholder_id', 'date'])['amount'].transform('sum')
df['txn_count_7d'] = df.groupby('cardholder_id')['amount'].transform(lambda x: x.rolling(7, min_periods=1).count())
df['merchant_visit_count'] = df.groupby(['cardholder_id', 'merchant_name']).cumcount()
df['is_new_merchant'] = (df['merchant_visit_count'] == 0).astype(int)
df['category_visit_count'] = df.groupby(['cardholder_id', 'merchant_category']).cumcount()
df['is_new_category'] = (df['category_visit_count'] == 0).astype(int)
cat_fraud_rate = df.groupby('merchant_category')['is_fraud'].mean()
df['category_risk_score'] = df['merchant_category'].map(cat_fraud_rate)
df['city_visit_count'] = df.groupby(['cardholder_id', 'merchant_city']).cumcount()
df['is_new_city'] = (df['city_visit_count'] == 0).astype(int)
ch_intl_rate = df.groupby('cardholder_id')['is_international'].transform(lambda x: x.expanding().mean().shift(1)).fillna(0)
df['intl_deviation'] = df['is_international'].astype(int) - ch_intl_rate
df['prev_country'] = df.groupby('cardholder_id')['merchant_country'].shift(1)
df['country_changed'] = (df['merchant_country'] != df['prev_country']).fillna(False).astype(int)
df['is_online_int'] = df['is_online'].astype(int)
ch_online_rate = df.groupby('cardholder_id')['is_online_int'].transform(lambda x: x.expanding().mean().shift(1)).fillna(0.5)
df['online_ratio_shift'] = df['is_online_int'] - ch_online_rate
df['amount_rolling_3d'] = df.groupby('cardholder_id')['amount'].transform(lambda x: x.rolling(3, min_periods=1).mean())
df['amount_rolling_7d'] = df.groupby('cardholder_id')['amount'].transform(lambda x: x.rolling(7, min_periods=1).mean())
df['spending_acceleration'] = df['amount_rolling_3d'] / (df['amount_rolling_7d'] + 1e-6)

le_cat = LabelEncoder()
df['merchant_category_encoded'] = le_cat.fit_transform(df['merchant_category'])
le_country = LabelEncoder()
df['merchant_country_encoded'] = le_country.fit_transform(df['merchant_country'])

feature_cols = [
    'hour', 'day_of_week', 'is_weekend', 'is_night',
    'time_since_last_txn', 'is_rapid',
    'amount', 'amount_log', 'ch_avg_amount', 'ch_std_amount',
    'amount_vs_personal_avg', 'is_round_amount',
    'daily_txn_count', 'daily_spend', 'txn_count_7d',
    'merchant_visit_count', 'is_new_merchant',
    'category_visit_count', 'is_new_category', 'category_risk_score',
    'city_visit_count', 'is_new_city',
    'intl_deviation', 'country_changed',
    'online_ratio_shift',
    'amount_rolling_3d', 'amount_rolling_7d', 'spending_acceleration',
    'merchant_category_encoded', 'merchant_country_encoded',
    'is_online_int', 'is_recurring', 'is_international',
]

for col in feature_cols:
    if df[col].dtype == 'bool':
        df[col] = df[col].astype(int)

X = df[feature_cols].fillna(0)
y = df['is_fraud']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
test_df = df.iloc[split_idx:]

scale = len(y_train[y_train==0]) / len(y_train[y_train==1])

model = xgb.XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.05,
    scale_pos_weight=scale, min_child_weight=3,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="aucpr", random_state=42, n_jobs=-1
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]
y_true = y_test.values

print(f"Model ready — AUC-PR: {average_precision_score(y_true, y_prob):.4f}")

# Now run bias checks
checker = BiasChecker()

# Bias across countries
country_result = checker.check_bias(
    y_true=y_true, y_pred=y_pred, y_prob=y_prob,
    group_labels=test_df['merchant_country'].values,
    group_name="Merchant Country"
)
checker.print_report(country_result)

# Bias across online vs in-store
channel_result = checker.check_bias(
    y_true=y_true, y_pred=y_pred, y_prob=y_prob,
    group_labels=test_df['is_online'].values,
    group_name="Transaction Channel"
)
checker.print_report(channel_result)

# Bias across merchant categories
category_result = checker.check_bias(
    y_true=y_true, y_pred=y_pred, y_prob=y_prob,
    group_labels=test_df['merchant_category'].values,
    group_name="Merchant Category"
)
checker.print_report(category_result)

Model ready — AUC-PR: 0.9359

BIAS ANALYSIS — Merchant Country

Group              Count  Fraud    Flag%      FPR      FNR   Recall
---------------------------------------------------------------
GB                97,391    364   0.375%   0.044%  11.539%    88.5%
IN                 1,276      0   0.000%   0.000%   0.000%     0.0%
US                 1,333     33   2.401%   0.154%   9.091%    90.9%

Disparate Impact Ratio: 0.1561 (threshold: >= 0.8)
  FAILED

Equalized Odds Max Diff: 0.1154 (threshold: <= 0.1)
  FPR diff: 0.0015
  FNR diff: 0.1154
  FAILED

OVERALL: *** FAILED — BLOCKED FROM PRODUCTION ***

BIAS ANALYSIS — Transaction Channel

Group              Count  Fraud    Flag%      FPR      FNR   Recall
---------------------------------------------------------------
False             64,050    219   0.334%   0.034%  12.329%    87.7%
True              35,950    178   0.509%   0.064%  10.112%    89.9%

Disparate Impact Ratio: 0.6564 (threshold: >= 0.8)
  FAILED

Equalized Odds Max D

In [6]:
%%writefile governance/explainability/shap_explainer.py
"""
SHAP Explainability Module
Enterprise MLOps Platform

Every prediction must be explainable.
"""

import shap
import numpy as np
import pandas as pd


class FraudExplainer:
    """Generates SHAP explanations for fraud predictions."""
    
    def __init__(self, model, feature_names: list):
        self.model = model
        self.feature_names = feature_names
        self.explainer = shap.TreeExplainer(model)
    
    def explain_prediction(self, features: np.ndarray) -> dict:
        """Explain a single prediction."""
        if features.ndim == 1:
            features = features.reshape(1, -1)
        
        shap_values = self.explainer.shap_values(features)
        
        # For binary classification, shap_values is for positive class
        if isinstance(shap_values, list):
            sv = shap_values[1][0]
        else:
            sv = shap_values[0]
        
        base_value = self.explainer.expected_value
        if isinstance(base_value, list):
            base_value = base_value[1]
        
        # Sort by absolute impact
        feature_impacts = []
        for i, (name, value) in enumerate(zip(self.feature_names, sv)):
            feature_impacts.append({
                "feature": name,
                "shap_value": round(float(value), 6),
                "feature_value": round(float(features[0][i]), 4),
                "direction": "fraud" if value > 0 else "legitimate",
                "abs_impact": abs(float(value)),
            })
        
        feature_impacts.sort(key=lambda x: x["abs_impact"], reverse=True)
        
        return {
            "base_value": round(float(base_value), 6),
            "prediction_shift": round(float(sum(sv)), 6),
            "top_factors": feature_impacts[:10],
            "all_factors": feature_impacts,
        }
    
    def explain_batch(self, X: np.ndarray, top_n: int = 5) -> list:
        """Explain multiple predictions."""
        shap_values = self.explainer.shap_values(X)
        
        if isinstance(shap_values, list):
            sv = shap_values[1]
        else:
            sv = shap_values
        
        explanations = []
        for i in range(len(X)):
            impacts = sorted(
                zip(self.feature_names, sv[i]),
                key=lambda x: abs(x[1]),
                reverse=True
            )[:top_n]
            
            explanations.append({
                "top_factors": [
                    {"feature": name, "shap_value": round(float(val), 6),
                     "direction": "fraud" if val > 0 else "legitimate"}
                    for name, val in impacts
                ]
            })
        
        return explanations
    
    def global_importance(self, X: np.ndarray) -> pd.DataFrame:
        """Global feature importance across all predictions."""
        shap_values = self.explainer.shap_values(X)
        
        if isinstance(shap_values, list):
            sv = shap_values[1]
        else:
            sv = shap_values
        
        importance = pd.DataFrame({
            "feature": self.feature_names,
            "mean_abs_shap": np.abs(sv).mean(axis=0),
        }).sort_values("mean_abs_shap", ascending=False)
        
        return importance

Writing governance/explainability/shap_explainer.py


In [7]:
from governance.explainability.shap_explainer import FraudExplainer
import numpy as np

# Create explainer
explainer = FraudExplainer(model, feature_cols)

# Explain a fraud prediction
fraud_indices = np.where((y_pred == 1) & (y_true == 1))[0]
legit_indices = np.where((y_pred == 0) & (y_true == 0))[0]

# Pick one caught fraud
fraud_idx = fraud_indices[0]
fraud_explanation = explainer.explain_prediction(X_test.iloc[fraud_idx].values)

print("FRAUD TRANSACTION — WHY WAS THIS FLAGGED?")
print(f"{'='*60}")
print(f"Fraud probability: {y_prob[fraud_idx]:.4f}")
print(f"\nTop factors pushing toward FRAUD:")
for f in fraud_explanation['top_factors'][:5]:
    if f['direction'] == 'fraud':
        print(f"  {f['feature']:<30} SHAP: +{f['shap_value']:.4f}  (value: {f['feature_value']})")

print(f"\nTop factors pushing toward LEGITIMATE:")
for f in fraud_explanation['top_factors'][:5]:
    if f['direction'] == 'legitimate':
        print(f"  {f['feature']:<30} SHAP: {f['shap_value']:.4f}  (value: {f['feature_value']})")

# Explain a legitimate transaction
legit_idx = legit_indices[0]
legit_explanation = explainer.explain_prediction(X_test.iloc[legit_idx].values)

print(f"\n\nLEGITIMATE TRANSACTION — WHY WAS THIS CLEARED?")
print(f"{'='*60}")
print(f"Fraud probability: {y_prob[legit_idx]:.6f}")
print(f"\nTop factors pushing toward LEGITIMATE:")
for f in legit_explanation['top_factors'][:5]:
    if f['direction'] == 'legitimate':
        print(f"  {f['feature']:<30} SHAP: {f['shap_value']:.4f}  (value: {f['feature_value']})")

# Global importance
print(f"\n\nGLOBAL FEATURE IMPORTANCE (SHAP)")
print(f"{'='*60}")
sample = X_test.sample(min(1000, len(X_test)), random_state=42)
global_imp = explainer.global_importance(sample.values)
for _, row in global_imp.head(15).iterrows():
    bar = "█" * int(row['mean_abs_shap'] * 100)
    print(f"  {row['feature']:<30} {row['mean_abs_shap']:.4f} {bar}")

FRAUD TRANSACTION — WHY WAS THIS FLAGGED?
Fraud probability: 0.9999

Top factors pushing toward FRAUD:
  amount                         SHAP: +2.5098  (value: 1770.59)
  daily_spend                    SHAP: +2.3518  (value: 3291.48)
  amount_rolling_7d              SHAP: +1.9435  (value: 261.0971)
  amount_log                     SHAP: +0.9600  (value: 7.4796)
  category_risk_score            SHAP: +0.8225  (value: 0.0237)

Top factors pushing toward LEGITIMATE:


LEGITIMATE TRANSACTION — WHY WAS THIS CLEARED?
Fraud probability: 0.000087

Top factors pushing toward LEGITIMATE:
  amount                         SHAP: -3.0786  (value: 14.96)
  daily_spend                    SHAP: -1.1709  (value: 14.96)
  amount_log                     SHAP: -1.1198  (value: 2.7701)
  amount_rolling_7d              SHAP: -1.0139  (value: 15.2386)
  amount_rolling_3d              SHAP: -0.9795  (value: 17.09)


GLOBAL FEATURE IMPORTANCE (SHAP)
  amount                         2.6655 ███████████████████████

In [8]:
%%writefile governance/audit/governance_gate.py
"""
Governance Gate — Production Approval Workflow
Enterprise MLOps Platform

No model reaches production without passing ALL gates.
"""

import json
from datetime import datetime


class GovernanceGate:
    """Automated governance checks before production deployment."""
    
    def __init__(self, min_auc_pr: float = 0.75, min_recall: float = 0.70):
        self.min_auc_pr = min_auc_pr
        self.min_recall = min_recall
        self.audit_log = []
    
    def _log(self, action: str, result: str, details: dict = None):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "action": action,
            "result": result,
            "details": details or {},
        }
        self.audit_log.append(entry)
        return entry
    
    def check_performance(self, metrics: dict) -> dict:
        """Gate 1: Model meets minimum performance thresholds."""
        auc_pr = metrics.get("auc_pr", 0)
        recall = metrics.get("recall", 0)
        
        passed = auc_pr >= self.min_auc_pr and recall >= self.min_recall
        
        result = {
            "gate": "performance",
            "passed": passed,
            "auc_pr": auc_pr,
            "auc_pr_threshold": self.min_auc_pr,
            "recall": recall,
            "recall_threshold": self.min_recall,
        }
        
        self._log("performance_check", "passed" if passed else "failed", result)
        return result
    
    def check_bias(self, bias_results: list) -> dict:
        """Gate 2: Model passes bias checks across all groups."""
        all_passed = all(r["overall_passed"] for r in bias_results)
        failed_groups = [r["group_name"] for r in bias_results if not r["overall_passed"]]
        
        result = {
            "gate": "bias",
            "passed": all_passed,
            "groups_checked": len(bias_results),
            "groups_failed": failed_groups,
        }
        
        self._log("bias_check", "passed" if all_passed else "failed", result)
        return result
    
    def check_explainability(self, global_importance, min_features: int = 5) -> dict:
        """Gate 3: Model predictions are explainable with sufficient feature diversity."""
        top_features = len(global_importance[global_importance['mean_abs_shap'] > 0.01])
        
        passed = top_features >= min_features
        
        result = {
            "gate": "explainability",
            "passed": passed,
            "meaningful_features": top_features,
            "min_required": min_features,
            "top_feature": global_importance.iloc[0]['feature'],
            "top_feature_dominance": round(
                global_importance.iloc[0]['mean_abs_shap'] / global_importance['mean_abs_shap'].sum(), 4
            ),
        }
        
        self._log("explainability_check", "passed" if passed else "failed", result)
        return result
    
    def check_model_card(self, card: dict) -> dict:
        """Gate 4: Model card exists with all required fields."""
        required_sections = ["model_details", "intended_use", "training_data",
                           "performance", "ethical_considerations", "limitations"]
        
        missing = [s for s in required_sections if s not in card]
        passed = len(missing) == 0
        
        result = {
            "gate": "model_card",
            "passed": passed,
            "missing_sections": missing,
        }
        
        self._log("model_card_check", "passed" if passed else "failed", result)
        return result
    
    def run_all_gates(self, metrics: dict, bias_results: list,
                      global_importance, model_card: dict) -> dict:
        """Run all governance gates and return final decision."""
        
        print(f"\nGOVERNANCE GATE — PRODUCTION APPROVAL")
        print(f"{'='*60}")
        print(f"Timestamp: {datetime.now().isoformat()}")
        
        g1 = self.check_performance(metrics)
        print(f"\n  Gate 1 — Performance:     {'PASSED' if g1['passed'] else 'FAILED'}")
        print(f"    AUC-PR: {g1['auc_pr']:.4f} (min: {g1['auc_pr_threshold']})")
        print(f"    Recall: {g1['recall']:.4f} (min: {g1['recall_threshold']})")
        
        g2 = self.check_bias(bias_results)
        print(f"\n  Gate 2 — Bias:            {'PASSED' if g2['passed'] else 'FAILED'}")
        print(f"    Groups checked: {g2['groups_checked']}")
        if g2['groups_failed']:
            print(f"    Failed groups: {', '.join(g2['groups_failed'])}")
        
        g3 = self.check_explainability(global_importance)
        print(f"\n  Gate 3 — Explainability:  {'PASSED' if g3['passed'] else 'FAILED'}")
        print(f"    Meaningful features: {g3['meaningful_features']} (min: {g3['min_required']})")
        print(f"    Top feature dominance: {g3['top_feature_dominance']*100:.1f}%")
        
        g4 = self.check_model_card(model_card)
        print(f"\n  Gate 4 — Model Card:      {'PASSED' if g4['passed'] else 'FAILED'}")
        if g4['missing_sections']:
            print(f"    Missing: {', '.join(g4['missing_sections'])}")
        
        all_passed = g1['passed'] and g2['passed'] and g3['passed'] and g4['passed']
        
        print(f"\n{'='*60}")
        if all_passed:
            print(f"  DECISION: APPROVED FOR PRODUCTION")
        else:
            failed_gates = []
            if not g1['passed']: failed_gates.append("performance")
            if not g2['passed']: failed_gates.append("bias")
            if not g3['passed']: failed_gates.append("explainability")
            if not g4['passed']: failed_gates.append("model_card")
            print(f"  DECISION: BLOCKED — failed gates: {', '.join(failed_gates)}")
        print(f"{'='*60}")
        
        decision = {
            "approved": all_passed,
            "timestamp": datetime.now().isoformat(),
            "gates": {
                "performance": g1,
                "bias": g2,
                "explainability": g3,
                "model_card": g4,
            },
            "audit_log": self.audit_log,
        }
        
        return decision
    
    def save_audit_log(self, path: str = "governance/audit/audit_log.json"):
        with open(path, 'w') as f:
            json.dump(self.audit_log, f, indent=2)
        print(f"Audit log saved: {path}")

Writing governance/audit/governance_gate.py


In [9]:
from governance.audit.governance_gate import GovernanceGate
from governance.model_cards.generator import generate_model_card

# Get model card
import mlflow
experiment = mlflow.get_experiment_by_name("fraud-detection-synthetic")
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.auc_pr DESC"])
best_run_id = runs.iloc[0]['run_id']
model_card = generate_model_card(best_run_id)

# Get metrics
metrics = {
    "auc_pr": average_precision_score(y_true, y_prob),
    "recall": (y_pred[y_true == 1] == 1).mean(),
}

# Run ALL governance gates
gate = GovernanceGate()
decision = gate.run_all_gates(
    metrics=metrics,
    bias_results=[country_result, channel_result, category_result],
    global_importance=global_imp,
    model_card=model_card,
)

gate.save_audit_log()


GOVERNANCE GATE — PRODUCTION APPROVAL
Timestamp: 2026-03-07T15:31:42.059904

  Gate 1 — Performance:     PASSED
    AUC-PR: 0.9359 (min: 0.75)
    Recall: 0.8866 (min: 0.7)

  Gate 2 — Bias:            FAILED
    Groups checked: 3
    Failed groups: Merchant Country, Transaction Channel, Merchant Category

  Gate 3 — Explainability:  PASSED
    Meaningful features: 29 (min: 5)
    Top feature dominance: 22.2%

  Gate 4 — Model Card:      PASSED

  DECISION: BLOCKED — failed gates: bias


TypeError: Object of type bool is not JSON serializable

In [10]:
# Fix the numpy bool serialization and save
import json
import numpy as np

class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.bool_, np.integer)):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

with open('governance/audit/audit_log.json', 'w') as f:
    json.dump(gate.audit_log, f, indent=2, cls=NumpyEncoder)

print("Audit log saved: governance/audit/audit_log.json")
print(f"\nAudit entries: {len(gate.audit_log)}")
for entry in gate.audit_log:
    print(f"  [{entry['result'].upper()}] {entry['action']}")

Audit log saved: governance/audit/audit_log.json

Audit entries: 4
  [PASSED] performance_check
  [FAILED] bias_check
  [PASSED] explainability_check
  [PASSED] model_card_check
